# Notebook Complementar -- Vaga Cientista de Dados 1 Itau 2026
## Sistemas de Recomendacao + Experimentacao + Deploy + Prova Estimada 2026

**Use junto com o Guia Definitivo v3.**

Este notebook cobre os gaps especificos da vaga de daily banking e marketplace:
- Modulo A: Sistemas de Recomendacao
- Modulo B: Experimentacao Estatistica e Testes A/B
- Modulo C: Deploy, Producao e Monitoramento
- Modulo D: Prova Estimada 2026 (40 questoes)

---

## Por que esses modulos?

A descricao da vaga diz explicitamente:
- 'problemas de personalizacao, recomendacoes e propensao'
- 'iniciativas de experimentacao, analises exploratorias e medicao de impacto'
- 'jornadas digitais hiperpersonalizadas'
- 'ambientes de cloud (AWS)'

Esses topicos nao estavam na prova de 2019, mas sao centrais para 2026.


---
# Setup
Execute antes de qualquer modulo.

In [ ]:
!pip install numpy pandas scikit-learn matplotlib seaborn scipy xgboost mlflow --quiet
print("OK!")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.sparse.linalg import svds
from scipy.linalg import svd
import warnings, os, math
warnings.filterwarnings('ignore')

from sklearn.metrics import mean_squared_error, ndcg_score
from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

np.random.seed(42)
plt.rcParams.update({'figure.figsize':(11,5), 'axes.grid':True, 'grid.alpha':0.3})
print("Imports OK!")

---
# MODULO A -- Sistemas de Recomendacao
## Relevancia para a vaga: MUITO ALTA (mencionado diretamente)

---

## A.1 O que e um sistema de recomendacao?

**Versao simples:** voce ja usou Netflix, Spotify, Amazon.
O sistema aprende o que voce gosta e sugere o que voce provavelmente vai gostar.

**No contexto do Itau (daily banking e marketplace):**
- Proximo melhor produto financeiro para oferecer (seguro, investimento, credito)
- Conteudo personalizado no app (notificacoes, ofertas contextuais)
- Ordem de exibicao de produtos no marketplace

**O problema formal:**
Dado um usuario u e um conjunto de itens I, prever a utilidade
de cada item para u e recomendar os k itens de maior utilidade.

---

## A.2 Tres abordagens principais

### 1. Filtragem Colaborativa (Collaborative Filtering)
**Ideia:** usuarios similares gostam de itens similares.
Usa o comportamento coletivo -- sem precisar de caracteristicas dos itens.

**User-based CF:** encontra usuarios parecidos com voce e recomenda o que eles gostaram.

**Item-based CF:** encontra itens parecidos com o que voce ja usou.
Mais estavel -- itens mudam menos que usuarios.

**Matrix Factorization (SVD):**
Decompoe a matriz usuario-item em fatores latentes.
Cada usuario e item e representado por um vetor de fatores.
A previsao e o produto interno dos dois vetores.

### 2. Filtragem Baseada em Conteudo (Content-Based)
**Ideia:** recomenda itens similares ao que voce ja consumiu,
baseado nas caracteristicas dos itens.
Nao precisa de dados de outros usuarios.
Limitacao: nao descobre novos tipos de produto.

### 3. Hibrido
Combina colaborativa com conteudo.
Usado por Netflix, Spotify, sistemas bancarios modernos.

---

## A.3 Problemas classicos

**Cold Start (inicio frio):**
- Usuario novo: sem historico, sem dados colaborativos.
- Item novo: sem avaliacoes, nao aparece na CF.
- Solucao: usar conteudo (demografico, perfil) para inicializar.

**Esparsidade:**
Matriz usuario-item e tipicamente 99%+ vazia.
A maioria dos usuarios interagiu com poucos itens.

**Escala:**
Milhoes de usuarios x milhoes de itens.
Algoritmos exatos sao inviaveis -- usar aproximacoes (ALS, SGD).

---

## A.4 Metricas de avaliacao de recomendacao

| Metrica | Formula | O que mede |
|---------|---------|------------|
| Precision@K | Relevantes nos K / K | Dos K recomendados, quantos sao relevantes? |
| Recall@K | Relevantes nos K / Total relevantes | Dos relevantes, quantos estao nos K? |
| NDCG@K | DCG@K / IDCG@K | Considera a POSICAO -- relevante no topo vale mais |
| MRR | mean(1/rank do 1o relevante) | Onde aparece o primeiro item relevante? |
| Coverage | % de itens recomendados / total | Diversidade do catalogo explorado |

**NDCG (Normalized Discounted Cumulative Gain):**
```
DCG@K = sum( relevancia_i / log2(i+1) ) para i=1 ate K
NDCG@K = DCG@K / DCG@K_ideal
```
Posicao 1 vale mais que posicao 2, que vale mais que posicao 3.
Metrica padrao em competicoes de recomendacao.

---

## A.5 Conexao com modelos de propensao

No Itau, recomendacao e frequentemente um modelo de **propensao**:
- P(usuario aceita produto X | perfil, contexto, historico)
- Ordena os produtos por probabilidade de aceitacao
- Recomenda o top-K para cada usuario

Isso conecta diretamente com regressao logistica e XGBoost que voce ja conhece.


In [ ]:
# A.1 -- Construindo um sistema de recomendacao do zero
# Cenario: recomendar produtos financeiros para clientes PJ

np.random.seed(42)

# Simulando matriz usuario-item (clientes x produtos)
n_clientes = 200
n_produtos  = 15
produtos = ['Conta PJ', 'Credito Giro', 'Seguro Empresarial', 'Investimento CDB',
            'Cartao PJ', 'Antecipacao Recebiveis', 'Seguro Frota', 'FGI',
            'Pix Cobranca', 'Maquininha', 'Folha Pagamento', 'Previdencia',
            'Cambio', 'Trade Finance', 'Garantia Bancaria']

# Matriz espars: a maioria e NaN (nao usou o produto)
matriz_raw = np.full((n_clientes, n_produtos), np.nan)

# Cada cliente usa em media 3-4 produtos
for i in range(n_clientes):
    n_usados = np.random.randint(2, 7)
    idx_usados = np.random.choice(n_produtos, n_usados, replace=False)
    for j in idx_usados:
        # Rating: 1-5 (satisfacao/engajamento)
        matriz_raw[i, j] = np.random.choice([3,4,5], p=[0.2,0.3,0.5])

# Estatisticas da matriz
total_cells = n_clientes * n_produtos
preenchidas = (~np.isnan(matriz_raw)).sum()
esparsidade = 1 - preenchidas/total_cells

print("=== MATRIZ USUARIO-ITEM ===")
print(f"  Dimensoes: {n_clientes} clientes x {n_produtos} produtos")
print(f"  Interacoes registradas: {preenchidas}")
print(f"  Esparsidade: {esparsidade*100:.1f}% (tipico: 95-99%)")
print()

# Visualizar a matriz
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Heatmap da matriz (primeiros 30 clientes)
sns.heatmap(pd.DataFrame(matriz_raw[:30], columns=produtos),
            ax=axes[0], cmap='YlOrRd', cbar_kws={'label':'Rating'},
            linewidths=0.1)
axes[0].set_title('Matriz Usuario-Item (30 primeiros clientes)\nBranco = nao usou (esparso)')
axes[0].set_xlabel('Produtos')
axes[0].set_ylabel('Clientes')
plt.setp(axes[0].get_xticklabels(), rotation=45, ha='right', fontsize=7)

# Distribuicao de uso por produto
uso_por_produto = (~np.isnan(matriz_raw)).sum(axis=0)
axes[1].barh(produtos, uso_por_produto, color='steelblue')
axes[1].set_title('Popularidade por produto\n(N de clientes que usaram)')
axes[1].set_xlabel('N de clientes')

plt.tight_layout()
plt.show()

print(f"Produto mais popular: {produtos[uso_por_produto.argmax()]} ({uso_por_produto.max()} clientes)")
print(f"Produto menos popular: {produtos[uso_por_produto.argmin()]} ({uso_por_produto.min()} clientes)")

In [ ]:
# A.2 -- Filtragem Colaborativa: User-based e Item-based

# Substituir NaN por 0 para calculos de similaridade
matriz = np.nan_to_num(matriz_raw, nan=0.0)

# Similaridade de Cosseno entre usuarios
def cosine_similarity_matrix(M):
    norms = np.linalg.norm(M, axis=1, keepdims=True)
    norms[norms == 0] = 1  # evitar divisao por zero
    M_norm = M / norms
    return M_norm @ M_norm.T

sim_usuarios = cosine_similarity_matrix(matriz)
sim_itens    = cosine_similarity_matrix(matriz.T)

print("=== FILTRAGEM COLABORATIVA USER-BASED ===")
print()

def recomendar_user_based(usuario_idx, matriz, sim_usuarios, n_rec=3, n_vizinhos=10):
    sims = sim_usuarios[usuario_idx].copy()
    sims[usuario_idx] = -1  # excluir o proprio usuario
    
    # Top K vizinhos mais similares
    top_vizinhos = np.argsort(sims)[::-1][:n_vizinhos]
    
    # Produtos que o usuario JA usou
    ja_usou = set(np.where(matriz[usuario_idx] > 0)[0])
    
    # Pontuacao ponderada pela similaridade
    scores = np.zeros(matriz.shape[1])
    for viz in top_vizinhos:
        scores += sims[viz] * (matriz[viz] > 0).astype(float)
    
    # Remover os que ja usou
    for j in ja_usou:
        scores[j] = -1
    
    top_recomendados = np.argsort(scores)[::-1][:n_rec]
    return top_recomendados, scores[top_recomendados]

# Testar para o cliente 0
cliente_teste = 0
ja_usou_idx = np.where(matriz[cliente_teste] > 0)[0]
rec_idx, rec_scores = recomendar_user_based(cliente_teste, matriz, sim_usuarios)

print(f"Cliente {cliente_teste}:")
print(f"  Produtos JA usados: {[produtos[i] for i in ja_usou_idx]}")
print(f"  Recomendacoes User-based:")
for idx, score in zip(rec_idx, rec_scores):
    print(f"    -> {produtos[idx]} (score={score:.3f})")

print()
print("=== FILTRAGEM COLABORATIVA ITEM-BASED ===")

def recomendar_item_based(usuario_idx, matriz, sim_itens, n_rec=3):
    ja_usou = np.where(matriz[usuario_idx] > 0)[0]
    scores = np.zeros(matriz.shape[1])
    
    for item_usado in ja_usou:
        scores += sim_itens[item_usado] * matriz[usuario_idx, item_usado]
    
    for j in ja_usou:
        scores[j] = -1
    
    top = np.argsort(scores)[::-1][:n_rec]
    return top, scores[top]

rec_item_idx, rec_item_scores = recomendar_item_based(cliente_teste, matriz, sim_itens)
print(f"  Recomendacoes Item-based:")
for idx, score in zip(rec_item_idx, rec_item_scores):
    print(f"    -> {produtos[idx]} (score={score:.3f})")

In [ ]:
# A.3 -- Matrix Factorization com SVD
print("=== MATRIX FACTORIZATION (SVD) ===")
print()

# SVD: decompoe a matriz em U * sigma * Vt
# U: fatores latentes dos usuarios
# Vt: fatores latentes dos itens
# sigma: importancia de cada fator

# Preencher NaN com media do usuario (necessario para SVD)
matriz_filled = matriz_raw.copy()
for i in range(n_clientes):
    linha = matriz_filled[i]
    media_usuario = np.nanmean(linha) if not np.all(np.isnan(linha)) else 3.0
    linha[np.isnan(linha)] = media_usuario
    matriz_filled[i] = linha

# SVD com k fatores latentes
k_fatores = 5
U, sigma, Vt = svds(matriz_filled, k=k_fatores)

# Variancia explicada por cada fator
variancia_total = np.sum(sigma**2)
variancia_por_fator = (sigma**2 / variancia_total * 100)[::-1]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(range(1, k_fatores+1), variancia_por_fator, color='steelblue')
axes[0].set_xlabel('Fator latente')
axes[0].set_ylabel('Variancia explicada (%)')
axes[0].set_title(f'SVD com {k_fatores} fatores latentes\n'
                   f'Fator 1 captura {variancia_por_fator[0]:.1f}% da variancia')

# Reconstrucao da matriz
matriz_reconstruida = U @ np.diag(sigma) @ Vt
# Clamp entre 1 e 5
matriz_reconstruida = np.clip(matriz_reconstruida, 1, 5)

# Comparar original vs reconstruida para usuario 0
original = matriz_raw[0]
reconstruida = matriz_reconstruida[0]
valid_mask = ~np.isnan(original)

axes[1].scatter(original[valid_mask], reconstruida[valid_mask], 
                alpha=0.7, s=80, color='coral')
axes[1].plot([1,5],[1,5], 'k--', lw=1.5, label='Perfeito')
axes[1].set_xlabel('Rating real'); axes[1].set_ylabel('Rating previsto (SVD)')
axes[1].set_title('SVD: ratings reais vs previstos (cliente 0)')
axes[1].legend()

plt.tight_layout()
plt.show()

# Recomendar usando SVD
def recomendar_svd(usuario_idx, matriz_raw, matriz_reconstruida, n_rec=3):
    ja_usou = set(np.where(~np.isnan(matriz_raw[usuario_idx]))[0])
    scores  = matriz_reconstruida[usuario_idx].copy()
    for j in ja_usou:
        scores[j] = -1
    top = np.argsort(scores)[::-1][:n_rec]
    return top, scores[top]

rec_svd, scores_svd = recomendar_svd(cliente_teste, matriz_raw, matriz_reconstruida)
print(f"Recomendacoes SVD para cliente {cliente_teste}:")
for idx, score in zip(rec_svd, scores_svd):
    print(f"  -> {produtos[idx]} (rating previsto: {score:.2f})")

print()
print("POR QUE SVD FUNCIONA:")
print("  Cada usuario e representado por um vetor de k 'gostos latentes'")
print("  Cada item e representado por um vetor de k 'caracteristicas latentes'")
print("  A previsao e o produto interno: quanto os gostos do usuario")
print("  se alinham com as caracteristicas do item")
print()
print("PROBLEMAS DO SVD CLASSICO:")
print("  - Esparsidade: SVD nao lida bem com muitos NaN")
print("  - Escala: para M usuarios x N itens, custo O(min(M,N)^3)")
print("  - Alternativas em producao: ALS (Alternating Least Squares),")
print("    SGD (Stochastic Gradient Descent), modelos neurais (two-tower)")

In [ ]:
# A.4 -- Metricas de avaliacao de recomendacao
print("=== METRICAS DE RECOMENDACAO ===")
print()

# Exemplo concreto
# Para o cliente 0, suponha que sabemos os itens relevantes (ground truth)
relevantes_reais = {2, 7, 11}  # itens que o cliente vai usar no futuro
k = 5

# Recomendacoes do modelo (ordenadas por score)
recomendacoes_modelo = [11, 3, 7, 14, 2]  # indices dos itens recomendados

# Precision@K
def precision_at_k(recomendados, relevantes, k):
    rec_k = recomendados[:k]
    hits  = sum(1 for r in rec_k if r in relevantes)
    return hits / k

# Recall@K
def recall_at_k(recomendados, relevantes, k):
    rec_k = recomendados[:k]
    hits  = sum(1 for r in rec_k if r in relevantes)
    return hits / len(relevantes) if relevantes else 0

# NDCG@K
def ndcg_at_k(recomendados, relevantes, k):
    rec_k = recomendados[:k]
    dcg   = sum(1/math.log2(i+2) for i, r in enumerate(rec_k) if r in relevantes)
    # DCG ideal: todos os relevantes no topo
    n_rel_in_k = min(len(relevantes), k)
    idcg  = sum(1/math.log2(i+2) for i in range(n_rel_in_k))
    return dcg/idcg if idcg > 0 else 0

# MRR
def mrr(recomendados, relevantes):
    for i, r in enumerate(recomendados):
        if r in relevantes:
            return 1/(i+1)
    return 0

prec = precision_at_k(recomendacoes_modelo, relevantes_reais, k)
rec  = recall_at_k(recomendacoes_modelo, relevantes_reais, k)
ndcg = ndcg_at_k(recomendacoes_modelo, relevantes_reais, k)
mrr_ = mrr(recomendacoes_modelo, relevantes_reais)

print(f"Recomendacoes (em ordem): {[produtos[i] for i in recomendacoes_modelo]}")
print(f"Itens relevantes reais:   {[produtos[i] for i in relevantes_reais]}")
print()
print(f"Precision@{k}: {prec:.4f}")
print(f"  Dos {k} recomendados, {prec*100:.0f}% eram relevantes")
print()
print(f"Recall@{k}:    {rec:.4f}")
print(f"  Dos {len(relevantes_reais)} relevantes reais, capturei {rec*100:.0f}%")
print()
print(f"NDCG@{k}:      {ndcg:.4f}")
print(f"  Item relevante na posicao 1 vale mais que na posicao 5")
print(f"  1.0 = ordem perfeita. Quanto maior, melhor.")
print()
print(f"MRR:           {mrr_:.4f}")
print(f"  Primeiro item relevante esta na posicao {int(1/mrr_)}")

print()
print("QUAL METRICA USAR?")
print("  Precision@K: quando cada recomendacao custa (ex: ligacao de vendas)")
print("  Recall@K:    quando perder um item relevante tem custo alto")
print("  NDCG@K:      quando a ORDER importa (ex: lista de produtos no app)")
print("  MRR:         quando so a PRIMEIRA recomendacao importa")

In [ ]:
# A.5 -- Modelo de propensao como sistema de recomendacao
print("=== PROPENSAO COMO RECOMENDACAO ===")
print()
print("No contexto bancario, recomendacao e frequentemente um modelo de propensao:")
print("P(cliente aceita produto X | perfil, contexto, historico)")
print()

# Simulando problema de propensao para produto 'Antecipacao de Recebiveis'
np.random.seed(42)
n = 2000

# Features do cliente PJ
df_prop = pd.DataFrame({
    'tempo_conta':    np.random.exponential(4, n),
    'volume_mensal':  np.random.lognormal(7, 1.5, n),
    'n_produtos':     np.random.poisson(3, n) + 1,
    'score_pj':       np.random.normal(650, 80, n).clip(300, 900),
    'sazonalidade':   np.random.choice([0,1], n, p=[0.6, 0.4]),
    'ja_tem_credito': np.random.binomial(1, 0.4, n),
    'inadim_90d':     np.random.binomial(1, 0.08, n),
})

# Target: aceitou a oferta de antecipacao
prob_aceite = 1 / (1 + np.exp(-(
    -3
    + 0.1  * df_prop['tempo_conta']
    + 0.3  * np.log1p(df_prop['volume_mensal'])
    + 0.2  * df_prop['n_produtos']
    + 0.005 * (df_prop['score_pj'] - 650)
    + 0.4  * df_prop['sazonalidade']
    + 0.3  * df_prop['ja_tem_credito']
    - 1.5  * df_prop['inadim_90d']
)))
df_prop['aceitou'] = np.random.binomial(1, prob_aceite)

print(f"Taxa de aceitacao: {df_prop['aceitou'].mean()*100:.1f}% (base desbalanceada)")
print()

# Modelo de propensao
X_prop = df_prop.drop('aceitou', axis=1)
y_prop = df_prop['aceitou']
X_tr_p, X_te_p, y_tr_p, y_te_p = train_test_split(X_prop, y_prop, test_size=0.2, random_state=42)

modelo_prop = XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=4,
                             verbosity=0, random_state=42)
modelo_prop.fit(X_tr_p, y_tr_p)

# Scores de propensao no teste
scores_prop = modelo_prop.predict_proba(X_te_p)[:,1]
auc_prop    = roc_auc_score(y_te_p, scores_prop)

print(f"Modelo XGBoost de propensao:")
print(f"  AUC: {auc_prop:.4f}")
print()

# Rankear clientes por propensao e calcular lift
df_resultado = pd.DataFrame({'score': scores_prop, 'aceitou': y_te_p.values})
df_resultado = df_resultado.sort_values('score', ascending=False).reset_index(drop=True)

# Lift curve: se abordarmos top K%, quantos aceites capturamos vs aleatorio?
percentuais = np.arange(5, 105, 5)
lift_vals   = []
for pct in percentuais:
    n_k    = int(len(df_resultado) * pct/100)
    taxa_k = df_resultado['aceitou'][:n_k].mean()
    taxa_g = df_resultado['aceitou'].mean()
    lift_vals.append(taxa_k / taxa_g if taxa_g > 0 else 1)

fig, axes = plt.subplots(1, 2, figsize=(14,5))
axes[0].plot(percentuais, lift_vals, 'b-o', ms=5, lw=2)
axes[0].axhline(1.0, color='gray', linestyle='--', label='Aleatorio (lift=1)')
axes[0].set_xlabel('% da base abordada')
axes[0].set_ylabel('Lift')
axes[0].set_title('Curva de Lift -- Modelo de Propensao\nQuanto melhor que aleatorio?')
axes[0].legend()

# Feature importance
fi_prop = pd.Series(modelo_prop.feature_importances_, index=X_prop.columns)
fi_prop.sort_values().plot(kind='barh', ax=axes[1], color='steelblue')
axes[1].set_title('Feature Importance -- Propensao Antecipacao')
plt.tight_layout(); plt.show()

print("LIFT na pratica:")
top10_lift = lift_vals[1]  # 10%
print(f"  Abordando o TOP 10% (maior propensao):")
print(f"  Taxa de aceite {top10_lift:.1f}x maior que abordagem aleatoria")
print(f"  Isso significa {top10_lift:.1f}x mais eficiente em custo de campanha")
print()
print("CONEXAO RECOMENDACAO <-> PROPENSAO:")
print("  Para cada cliente: calcula propensao para cada produto disponivel")
print("  Ordena por propensao: P(aceita produto X), P(aceita Y), ...")
print("  Recomenda o top-K: 'proximo melhor produto' (Next Best Product)")

In [ ]:
# A.6 -- Cold Start e Exercicios
print("=== COLD START -- O PROBLEMA DO USUARIO/ITEM NOVO ===")
print()
print("CENARIO: novo cliente PJ abre conta hoje.")
print("Sem historico de transacoes, sem produtos contratados.")
print("Como recomendar o primeiro produto?")
print()
print("SOLUCOES PARA COLD START:")
solucoes = [
    ("Content-based com perfil demografico",
     "Usar CNAE, porte, regiao, faturamento declarado para segmentar",
     "Imediato, mas requer dados do cadastro"),
    ("Regras de negocio + dados externos",
     "Ex: empresa do setor alimenticio -> recomendar maquininha",
     "Rapido, interpretavel, mas manual"),
    ("Modelo de propensao com features de cadastro",
     "Treinar modelo usando apenas features disponiveis no cadastro",
     "Requer dados historicos de outros clientes similares"),
    ("Popular items (baseline)",
     "Recomendar os produtos mais populares do segmento",
     "Simples, funciona razoavelmente, zero personalizacao"),
    ("Onboarding ativo",
     "Perguntar ao cliente o que precisa (pesquisa de necessidades)",
     "Alta qualidade, mas gera atrito"),
]
for nome, descricao, obs in solucoes:
    print(f"  {nome}:")
    print(f"    Como: {descricao}")
    print(f"    Observacao: {obs}")
    print()

print("="*65)
print("EXERCICIOS -- SISTEMAS DE RECOMENDACAO")
print("="*65)
print("""
E1: O que e esparsidade em sistemas de recomendacao?
    Por que e um problema? Como matrix factorization ajuda?

E2: Diferenca entre Precision@K e NDCG@K.
    Quando voce usaria cada uma?

E3: Um cliente novo abriu conta no Itau hoje.
    O sistema de recomendacao colaborativa nao consegue recomendar nada.
    Como voce resolveria o cold start?

E4: Qual a diferenca entre filtragem colaborativa e content-based?
    Cite uma vantagem e desvantagem de cada.

E5: Como um modelo de propensao (XGBoost) pode ser usado como
    sistema de recomendacao? Descreva o pipeline completo.
""")

print("GABARITOS:")
print("""
E1: Esparsidade = a maior parte da matriz usuario-item e vazia.
    Clientes interagiram com poucos produtos (3-5 de 50+).
    Problema: sem dados suficientes para calcular similaridade confiavel.
    Matrix Factorization: aprende representacoes latentes mesmo com esparsidade.
    SVD encontra k fatores que melhor explicam as interacoes observadas.
    Generaliza para pares (usuario, item) nunca vistos.

E2: Precision@K: dos K recomendados, quantos sao relevantes?
    Nao considera a ordem dentro dos K.
    Use quando: cada recomendacao tem custo fixo (ex: ligacao de vendas).
    
    NDCG@K: item relevante na posicao 1 vale mais que na posicao 5.
    Considera a ordem -- mais rigorosa e mais realista.
    Use quando: a ordem importa (lista no app, primeiros itens tem mais cliques).
    Padrao em competicoes de recomendacao.

E3: Estrategias para cold start:
    a) Segmentar por CNAE e porte: empresa alimenticia -> maquininha, folha
    b) Usar modelo de propensao com features de cadastro (sem historico)
    c) Popular items do segmento como baseline
    d) Perguntar ao cliente durante onboarding (higher friction, higher quality)
    Na pratica: combinar a+b como modelo hibrido.

E4: Colaborativa: usa comportamento coletivo. Sem caracteristicas dos itens.
    Vantagem: descobre preferencias inesperadas (serendipidade).
    Desvantagem: cold start, escala, esparsidade.
    
    Content-based: usa caracteristicas dos itens. Sem outros usuarios.
    Vantagem: sem cold start de item, personalizacao imediata.
    Desvantagem: nao descobre novos tipos, bolha de filtro.

E5: Pipeline de Next Best Product (propensao como recomendacao):
    1. Para cada cliente: calcular features (perfil, historico, contexto)
    2. Para cada produto disponivel: rodar XGBoost -> P(aceita produto)
    3. Filtrar produtos ja contratados e inelegiveis (restricoes de credito)
    4. Ordenar por probabilidade: [produto A: 0.72, B: 0.45, C: 0.31]
    5. Recomendar top-K: exibir no app ou acionar campanha
    6. Avaliar com Precision@K e lift vs baseline aleatorio
    7. Retreinar periodicamente com novos dados de resposta
""")

---
# MODULO B -- Experimentacao Estatistica e Testes A/B
## Relevancia para a vaga: MUITO ALTA ("apoiar iniciativas de experimentacao")

---

## B.1 Por que experimentacao?

**Problema de inferencia causal:** correlacao nao e causalidade.
Para saber se uma mudanca CAUSOU uma melhoria, precisamos de experimento controlado.

**Exemplo:** nova tela de onboarding aumentou o NPS?
- Observacional: clientes que usaram a nova tela tem NPS maior. Mas... pode ser
  que clientes mais engajados (que exploram o app) tanto usem a nova tela
  quanto tenham NPS maior. Nao sabemos a causa.
- Experimental (A/B test): atribuicao ALEATORIA. Grupo A: nova tela.
  Grupo B: tela antiga. A aleatoriedade garante que os grupos sao comparaveis.

---

## B.2 Estrutura de um teste A/B

1. **Hipotese:** nova tela de onboarding aumenta o NPS
2. **Metrica primaria:** NPS medio (o que queremos melhorar)
3. **Metricas de guardrail:** taxa de abandono, tempo de sessao (nao podemos piorar)
4. **Tamanho de amostra:** calcular ANTES de comecar o teste
5. **Duracao:** definir ANTES de comecar (sem peeking!)
6. **Randomizacao:** garantir que A e B sao comparaveis
7. **Analise:** teste estatistico + tamanho do efeito + IC
8. **Decisao:** baseada em criterio pre-definido, nao em p-valor isolado

---

## B.3 Calculo do tamanho de amostra

Depende de tres fatores:
- **alpha:** nivel de significancia (geralmente 0.05). Taxa de erro tipo 1.
- **beta:** poder do teste (geralmente 0.80). 1-beta = poder.
- **MDE:** minimo efeito detectavel. Qual o menor efeito que importa para o negocio?

Formula aproximada para duas proporcoes:
```
n = 2 * (z_alpha/2 + z_beta)^2 * p*(1-p) / delta^2
```
onde delta e o MDE e p e a taxa de conversao baseline.

**Erro classico:** comecar o teste sem calcular o tamanho de amostra.
Resultado: teste sem poder suficiente para detectar efeitos reais.

---

## B.4 O problema do Peeking (olhar antes do tempo)

Se voce checa os resultados toda hora e para quando p < 0.05,
a taxa de erro tipo 1 REAL e muito maior que 5%.

**Por que:** com muitas checagens, e provavel que em algum momento
os dados mostrem p < 0.05 por acaso, mesmo sem efeito real.

**Solucao:**
- Definir a duracao ANTES de comecar
- So olhar ao final
- Ou usar Sequential Testing (SPRT, always-valid p-values)

---

## B.5 Quando A/B Test nao e possivel

| Metodo | Quando usar | Exigencia |
|--------|-------------|-----------|
| Diferencias em Diferencias (DiD) | Grupo tratado e controle com dados de antes e depois | Tendencias paralelas pre-tratamento |
| Regressao Descontínua (RDD) | Tratamento determinado por threshold | Dados proximos ao threshold |
| Variaveis Instrumentais | Confundidor nao observado | Instrumento valido |
| Propensity Score Matching | Dados observacionais com muitas covariadas | Ignorabilidade condicional |


In [ ]:
# B.1 -- Calculo de tamanho de amostra e poder do teste
print("=== CALCULO DO TAMANHO DE AMOSTRA ===")
print()

from scipy.stats import norm

def tamanho_amostra_proporcoes(p_baseline, mde, alpha=0.05, poder=0.80):
    # p_baseline: taxa de conversao no grupo controle
    # mde: minimo efeito detectavel (diferenca absoluta)
    # alpha: nivel de significancia
    # poder: poder do teste (1 - beta)
    z_alpha = norm.ppf(1 - alpha/2)  # teste bicaudal
    z_beta  = norm.ppf(poder)
    
    p1 = p_baseline
    p2 = p_baseline + mde
    p_pooled = (p1 + p2) / 2
    
    n = (z_alpha + z_beta)**2 * (p1*(1-p1) + p2*(1-p2)) / (mde**2)
    return math.ceil(n)

# Cenario: nova oferta de credito no app
# Baseline: 8% de clientes aceitam a oferta atual
# Queremos detectar melhora de pelo menos 2pp (de 8% para 10%)
p_base = 0.08
mde    = 0.02  # 2 pontos percentuais

n_necessario = tamanho_amostra_proporcoes(p_base, mde)
print(f"Cenario: taxa de aceite baseline = {p_base*100:.0f}%")
print(f"Efeito minimo que importa (MDE) = {mde*100:.0f}pp")
print(f"Tamanho de amostra necessario (por grupo): {n_necessario:,}")
print(f"Total (A + B): {n_necessario*2:,} clientes")
print()

# Tabela de tamanhos para diferentes MDEs e poderes
print("TAMANHO DE AMOSTRA POR GRUPO (alpha=5%, baseline=8%):")
print(f"{'MDE':>8} {'Poder 70%':>12} {'Poder 80%':>12} {'Poder 90%':>12}")
print("-"*48)
for mde_pct in [0.5, 1.0, 1.5, 2.0, 3.0, 5.0]:
    mde_val = mde_pct / 100
    n70 = tamanho_amostra_proporcoes(p_base, mde_val, poder=0.70)
    n80 = tamanho_amostra_proporcoes(p_base, mde_val, poder=0.80)
    n90 = tamanho_amostra_proporcoes(p_base, mde_val, poder=0.90)
    print(f"{mde_pct:>7.1f}%  {n70:>12,} {n80:>12,} {n90:>12,}")

print()
print("INTERPRETACAO:")
print("  MDE menor = mais sensivel = precisa de MAIS amostra")
print("  Poder maior = detecta efeitos menores = precisa de MAIS amostra")
print("  Escolha o MDE baseado no impacto minimo relevante para o negocio")
print("  Nao adianta detectar diferenca de 0.1% se ela nao tem impacto real")

In [ ]:
# B.2 -- Executando e analisando um teste A/B
np.random.seed(42)

# Cenario: novo fluxo de onboarding no app do Itau
# Grupo A (controle): fluxo atual
# Grupo B (tratamento): novo fluxo simplificado

# Parametros reais do experimento
n_por_grupo = 5000
p_controle  = 0.082   # 8.2% completam o onboarding no controle
p_tratamento = 0.098  # 9.8% completam no tratamento (efeito real: +1.6pp)

# Simulando resultados
controle   = np.random.binomial(1, p_controle,  n_por_grupo)
tratamento = np.random.binomial(1, p_tratamento, n_por_grupo)

# Teste de proporcoes (qui-quadrado ou z-test)
from scipy.stats import chi2_contingency, ttest_ind

n_ctrl_sucesso = controle.sum()
n_trat_sucesso = tratamento.sum()

# Tabela de contingencia
tabela = np.array([[n_ctrl_sucesso,  n_por_grupo - n_ctrl_sucesso],
                   [n_trat_sucesso,  n_por_grupo - n_trat_sucesso]])

chi2, p_value, dof, expected = chi2_contingency(tabela)

taxa_ctrl = controle.mean()
taxa_trat = tratamento.mean()
diferenca = taxa_trat - taxa_ctrl

# Intervalo de confianca da diferenca
se_diff = np.sqrt(taxa_ctrl*(1-taxa_ctrl)/n_por_grupo +
                  taxa_trat*(1-taxa_trat)/n_por_grupo)
ic_95   = (diferenca - 1.96*se_diff, diferenca + 1.96*se_diff)

print("="*65)
print("RESULTADO DO TESTE A/B -- Novo onboarding")
print("="*65)
print()
print(f"GRUPO CONTROLE (fluxo atual):")
print(f"  N: {n_por_grupo:,} | Conversoes: {n_ctrl_sucesso:,} | Taxa: {taxa_ctrl*100:.2f}%")
print()
print(f"GRUPO TRATAMENTO (novo fluxo):")
print(f"  N: {n_por_grupo:,} | Conversoes: {n_trat_sucesso:,} | Taxa: {taxa_trat*100:.2f}%")
print()
print(f"RESULTADO:")
print(f"  Diferenca absoluta: {diferenca*100:+.2f}pp")
print(f"  IC 95%: [{ic_95[0]*100:.2f}pp, {ic_95[1]*100:.2f}pp]")
print(f"  Estatistica chi^2: {chi2:.4f}")
print(f"  P-valor: {p_value:.4f}")
print()

alpha = 0.05
if p_value < alpha:
    print(f"p={p_value:.4f} < {alpha} -> RESULTADO ESTATISTICAMENTE SIGNIFICATIVO")
    print("Recomendacao: adotar o novo fluxo de onboarding")
else:
    print(f"p={p_value:.4f} >= {alpha} -> Sem evidencia suficiente de diferenca")

print()
print("INTERPRETACAO CORRETA:")
print(f"  O novo fluxo AUMENTOU a taxa de conversao em {diferenca*100:.2f}pp")
print(f"  IC 95% [{ic_95[0]*100:.2f}pp, {ic_95[1]*100:.2f}pp] nao inclui zero")
print(f"  -> Ha evidencia estatistica de melhora real")
print()
print("IMPACTO DE NEGOCIO:")
clientes_mes = 50000
conversoes_extra = int(clientes_mes * diferenca)
print(f"  Com {clientes_mes:,} novos clientes/mes:")
print(f"  Conversoes extras: ~{conversoes_extra:,} clientes/mes")

In [ ]:
# B.3 -- O problema do Peeking
print("=== PROBLEMA DO PEEKING ===")
print()
print("O que e: checar os resultados todo dia e parar quando p < 0.05")
print("Por que e errado: infla a taxa de falso positivo (erro tipo 1)")
print()

# Simulacao: sem efeito real, quanto tempo ate p < 0.05 por acaso?
np.random.seed(42)
n_simulacoes = 1000
p_verdadeiro = 0.10  # sem diferenca real entre os grupos

falsos_positivos_peeking = 0
falsos_positivos_correto = 0
n_maximo = 2000

for sim in range(n_simulacoes):
    controle_cum   = []
    tratamento_cum = []
    encontrou_sig  = False

    for t in range(10, n_maximo+1, 10):
        c = np.random.binomial(1, p_verdadeiro, 10)
        tr = np.random.binomial(1, p_verdadeiro, 10)
        controle_cum.extend(c)
        tratamento_cum.extend(tr)

        if t < n_maximo:  # peeking: checa antes do tempo
            nc, nt = np.array(controle_cum), np.array(tratamento_cum)
            if len(nc) >= 20:
                from scipy.stats import chi2_contingency
                tab = np.array([[nc.sum(), len(nc)-nc.sum()],
                                [nt.sum(), len(nt)-nt.sum()]])
                try:
                    _, p, _, _ = chi2_contingency(tab)
                    if p < 0.05 and not encontrou_sig:
                        falsos_positivos_peeking += 1
                        encontrou_sig = True
                except:
                    pass

    # Analise correta: so ao final
    nc = np.array(controle_cum[:n_maximo])
    nt = np.array(tratamento_cum[:n_maximo])
    tab = np.array([[nc.sum(), len(nc)-nc.sum()],
                    [nt.sum(), len(nt)-nt.sum()]])
    try:
        _, p_final, _, _ = chi2_contingency(tab)
        if p_final < 0.05:
            falsos_positivos_correto += 1
    except:
        pass

taxa_fp_peeking = falsos_positivos_peeking / n_simulacoes
taxa_fp_correto = falsos_positivos_correto / n_simulacoes

print(f"Simulacao: {n_simulacoes} experimentos SEM efeito real (p=10% em ambos)")
print(f"  Taxa de falso positivo com PEEKING:  {taxa_fp_peeking*100:.1f}% (deveria ser 5%)")
print(f"  Taxa de falso positivo CORRETO:       {taxa_fp_correto*100:.1f}% (correto!)")
print()
print("CONCLUSAO: peeking infla severamente a taxa de falso positivo!")
print("Com peeking diario, voce pode ter ate 3-5x mais falsos positivos")
print("do que o alpha nominal de 5%.")
print()
print("COMO EVITAR:")
print("  1. Calcule o tamanho de amostra ANTES de comecar")
print("  2. Defina a duracao do teste ANTES de comecar")
print("  3. So analise ao final da duracao pre-definida")
print("  4. Alternativa: Sequential Testing (SPRT) para monitoramento continuo")

In [ ]:
# B.4 -- Diferencas em Diferencas (DiD)
print("=== DIFERENCAS EM DIFERENCAS (DiD) ===")
print()
print("Quando usar: A/B test nao e possivel. Voce implementou algo")
print("em uma regiao/segmento e quer medir o efeito causalmente.")
print()

np.random.seed(42)
n = 200

# Cenario: nova politica de atendimento implementada em SP (tratado)
# RJ e MG nao receberam a politica (controle)

# Tendencia temporal comum (antes da politica)
tempo = np.arange(12)  # 12 meses

# Grupo SP (tratado): politica implementada no mes 6
sp_antes = 7.0 + 0.1*tempo[:6] + np.random.normal(0, 0.2, 6)
sp_depois = 7.6 + 0.1*tempo[6:] + np.random.normal(0, 0.2, 6)  # sobe apos politica
sp_nps = np.concatenate([sp_antes, sp_depois])

# Grupo RJ+MG (controle): sem politica, tendencia paralela
rj_nps = 6.8 + 0.1*tempo + np.random.normal(0, 0.2, 12)

# Diferenca em Diferencas
media_sp_antes  = sp_nps[:6].mean()
media_sp_depois = sp_nps[6:].mean()
media_rj_antes  = rj_nps[:6].mean()
media_rj_depois = rj_nps[6:].mean()

did = (media_sp_depois - media_sp_antes) - (media_rj_depois - media_rj_antes)

fig, axes = plt.subplots(1, 2, figsize=(14,5))

axes[0].plot(tempo[:6], sp_nps[:6],  'b-o', ms=6, label='SP (tratado) - antes')
axes[0].plot(tempo[6:], sp_nps[6:],  'b-s', ms=6, label='SP (tratado) - depois')
axes[0].plot(tempo,     rj_nps,      'r-o', ms=6, label='RJ+MG (controle)')
axes[0].axvline(5.5, color='gray', linestyle='--', lw=2, label='Implementacao da politica')
axes[0].set_xlabel('Mes'); axes[0].set_ylabel('NPS medio')
axes[0].set_title('Diferencas em Diferencas')
axes[0].legend()

# Visualizando o DiD
ax2 = axes[1]
grupos = ['Controle
(antes)', 'Controle
(depois)', 'Tratado
(antes)', 'Tratado
(depois)']
valores = [media_rj_antes, media_rj_depois, media_sp_antes, media_sp_depois]
cores   = ['lightcoral','coral','lightblue','steelblue']
bars = ax2.bar(grupos, valores, color=cores, edgecolor='black', width=0.5)
ax2.set_ylabel('NPS medio')
ax2.set_title(f'Decomposicao do DiD\n'
               f'Delta SP={media_sp_depois-media_sp_antes:.2f} | '
               f'Delta RJ={media_rj_depois-media_rj_antes:.2f} | '
               f'DiD={did:.2f}')
# Anotacoes
ax2.annotate('', xy=(1, media_rj_depois), xytext=(0, media_rj_antes),
             arrowprops=dict(arrowstyle='<->', color='red'))
ax2.annotate('', xy=(3, media_sp_depois), xytext=(2, media_sp_antes),
             arrowprops=dict(arrowstyle='<->', color='blue'))

plt.tight_layout(); plt.show()

print(f"RESULTADOS DiD:")
print(f"  SP (tratado):    antes={media_sp_antes:.2f} | depois={media_sp_depois:.2f} | delta={media_sp_depois-media_sp_antes:.2f}")
print(f"  RJ+MG (controle):antes={media_rj_antes:.2f} | depois={media_rj_depois:.2f} | delta={media_rj_depois-media_rj_antes:.2f}")
print(f"  Efeito causal estimado (DiD): {did:.2f} pontos de NPS")
print()
print("PREMISSA CRITICA DO DiD: Tendencias Paralelas")
print("  Na ausencia da politica, SP e RJ teriam evoluido IDENTICAMENTE")
print("  Verificar: plotar as series ANTES do tratamento -- devem ser paralelas")
print("  Se SP ja estava crescendo mais que RJ antes, o DiD sera enviesado")

In [ ]:
# B.5 -- Exercicios Experimentacao
print("="*65)
print("EXERCICIOS -- EXPERIMENTACAO E TESTES A/B")
print("="*65)
print("""
E1: Voce quer testar se um novo modelo de recomendacao aumenta a taxa
    de clique de 5% para 6%. Alpha=5%, Poder=80%.
    Quanto de amostra voce precisa por grupo?

E2: Voce implementou um teste A/B. Apos 3 dias (n=1000 por grupo),
    p-valor = 0.03. Voce para o teste e declara vitoria?

E3: O que sao metricas de guardrail? Cite 2 exemplos para um teste
    de nova tela de checkout no app.

E4: Por que A/B test e considerado o 'padrao ouro' para inferencia causal?

E5: Quando voce usaria DiD em vez de A/B test?
""")

from scipy.stats import norm
def n_amostra(p_base, mde, alpha=0.05, poder=0.80):
    z_a = norm.ppf(1-alpha/2); z_b = norm.ppf(poder)
    p2  = p_base + mde
    return math.ceil((z_a+z_b)**2 * (p_base*(1-p_base)+p2*(1-p2)) / mde**2)

n_e1 = n_amostra(0.05, 0.01)
print("GABARITOS:")
print(f"""
E1: n = {n_e1:,} clientes por grupo (total: {n_e1*2:,})
    Formula: n = (z_alpha/2 + z_beta)^2 * (p1(1-p1) + p2(1-p2)) / delta^2
    Com p1=5%, p2=6% (MDE=1pp), alpha=5%, poder=80%.

E2: NAO. Isso e o problema do PEEKING.
    Voce definiu que precisava de X clientes por grupo (calculado antes).
    Parar antes porque p<0.05 infla a taxa de falso positivo para 20-30%+.
    Continue ate atingir o tamanho pre-definido.
    Alternativa correta: use Sequential Testing se precisar monitorar.

E3: Metricas de guardrail sao metricas que voce NAO pode piorar.
    Mesmo que a metrica primaria melhore, se uma guardrail piorar,
    o teste falhou. Exemplos para checkout:
    - Taxa de abandono do app (nao pode subir)
    - Tempo medio de sessao (nao pode cair)
    - Taxa de erros tecnicos (nao pode subir)
    - NPS no feedback (nao pode cair)

E4: Randomizacao garante que A e B sao estatisticamente equivalentes.
    Isso controla TODOS os confundidores -- observados e nao-observados.
    Com randomizacao, qualquer diferenca observada e causal (ou acaso).
    Metodos observacionais (DiD, RDD) sempre dependem de premissas nao-verificaveis.

E5: DiD quando A/B nao e possivel:
    - Politica regulatoria que afeta toda a carteira (nao da pra ter controle)
    - Restricao legal ou etica de randomizar (ex: produto de seguro)
    - Mudanca ja implementada -- quer medir o efeito retroativamente
    - Custo de randomizar e alto (ex: mudanca em sistema legado)
    Exige: grupo de controle observavel, dados antes e depois, tendencias paralelas.
""")

---
# MODULO C -- Deploy, Producao e Monitoramento
## Relevancia para a vaga: MEDIA (nao e o foco de Cientista 1, mas aparece na entrevista)

---

## C.1 O ciclo de vida de um modelo em producao

```
Problema de negocio
       |
  Dados + Feature Engineering
       |
  Experimentacao (MLflow)
       |
  Validacao (CV, holdout temporal)
       |
  Deploy (endpoint, batch job)
       |
  Monitoramento (drift, metricas de negocio)
       |
  Retreino (quando necessario)
       |
  (volta para Experimentacao)
```

---

## C.2 Tipos de deploy

| Tipo | Como funciona | Quando usar |
|------|--------------|-------------|
| **Real-time endpoint** | Previsao sob demanda (< 100ms) | Recomendacao no app em tempo real |
| **Batch scoring** | Roda offline, gera arquivo de scores | Propensao diaria para campanha |
| **Streaming** | Processa eventos em tempo real | Deteccao de fraude em transacao |

No Itau: modelos de propensao geralmente rodam em **batch** (diario/semanal).
Modelos de fraude precisam de **real-time** (milissegundos).

---

## C.3 Monitoramento de drift

**Por que modelos degradam em producao:**
- Distribuicao dos dados muda com o tempo (data drift)
- Relacao entre X e Y muda (concept drift)
- Comportamento do usuario muda (ex: pandemia, crise economica)

**Tipos de drift:**

| Tipo | O que muda | Detectado por |
|------|-----------|---------------|
| Data drift | Distribuicao de X | Testes estatisticos (KS, PSI) |
| Concept drift | Relacao X->Y | Queda nas metricas de performance |
| Label drift | Distribuicao de Y | Monitoramento do target |

**PSI (Population Stability Index):**
Mede o quanto a distribuicao de uma variavel mudou entre treino e producao.
- PSI < 0.1: mudanca insignificante
- PSI 0.1-0.25: mudanca moderada (investigar)
- PSI > 0.25: mudanca severa (retreinar o modelo)

---

## C.4 SageMaker no contexto de ML

Servico AWS para treino e deploy de modelos de ML em escala.

**Componentes principais:**
- **SageMaker Studio:** ambiente de notebook gerenciado
- **Training Jobs:** treino distribuido em instancias gerenciadas
- **Endpoints:** deploy de modelos para inferencia real-time
- **Batch Transform:** scoring em batch de grandes volumes
- **Model Monitor:** monitoramento automatico de drift
- **Experiments:** tracking similar ao MLflow
- **Pipelines:** orquestracao do ciclo completo

**Para o Itau:** dados ficam no S3, processamento no Athena,
treino no SageMaker Training Jobs, serve via Endpoint ou Batch.


In [ ]:
# C.1 -- Pipeline completo de producao (simulado)
import mlflow, mlflow.sklearn
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

print("=== PIPELINE COMPLETO DE ML PARA PRODUCAO ===")
print()

np.random.seed(42)

# 1. DADOS: simular dados de treino (historico) e producao (dados novos)
def gerar_dados(n, drift=False):
    # Gera dados de clientes PJ para modelo de propensao
    df = pd.DataFrame({
        'tempo_conta':    np.random.exponential(4, n),
        'volume_mensal':  np.random.lognormal(7 + (0.5 if drift else 0), 1.5, n),
        'n_produtos':     np.random.poisson(3, n) + 1,
        'score_pj':       np.random.normal(650 + (30 if drift else 0), 80, n),
        'sazonalidade':   np.random.binomial(1, 0.4, n),
    })
    # Target
    prob = 1/(1+np.exp(-(
        -2 + 0.3*np.log1p(df['volume_mensal']) +
        0.2*df['n_produtos'] + 0.004*(df['score_pj']-650))))
    df['target'] = np.random.binomial(1, prob)
    return df

df_treino = gerar_dados(5000)
df_producao_ok   = gerar_dados(2000, drift=False)  # sem drift
df_producao_drift= gerar_dados(2000, drift=True)   # com drift

print("1. DADOS:")
print(f"   Treino: {len(df_treino):,} | Producao OK: {len(df_producao_ok):,}")
print()

# 2. TREINO com MLflow
mlflow.set_experiment('producao_propensao_v1')

features = ['tempo_conta','volume_mensal','n_produtos','score_pj','sazonalidade']
X_tr = df_treino[features]; y_tr = df_treino['target']
X_te = df_producao_ok[features]

with mlflow.start_run(run_name='xgb_v1'):
    pipe_prod = Pipeline([
        ('sc', StandardScaler()),
        ('xgb', XGBClassifier(n_estimators=200, learning_rate=0.05,
                              max_depth=4, verbosity=0, random_state=42))
    ])
    pipe_prod.fit(X_tr, y_tr)
    
    auc_treino = roc_auc_score(y_tr, pipe_prod.predict_proba(X_tr)[:,1])
    auc_prod   = roc_auc_score(df_producao_ok['target'],
                               pipe_prod.predict_proba(X_te)[:,1])
    
    mlflow.log_param('modelo', 'XGBoost')
    mlflow.log_param('n_estimators', 200)
    mlflow.log_metric('auc_treino', round(auc_treino, 4))
    mlflow.log_metric('auc_producao', round(auc_prod, 4))
    mlflow.sklearn.log_model(pipe_prod, 'modelo_propensao')

print("2. TREINO:")
print(f"   AUC treino:   {auc_treino:.4f}")
print(f"   AUC producao: {auc_prod:.4f}  (sem drift)")
print()

# 3. MONITORAMENTO DE DRIFT (PSI)
def calcular_psi(referencia, atual, n_bins=10):
    # Population Stability Index
    bins = np.percentile(referencia, np.linspace(0, 100, n_bins+1))
    bins[0] = -np.inf; bins[-1] = np.inf
    
    freq_ref   = np.histogram(referencia, bins=bins)[0] / len(referencia)
    freq_atual = np.histogram(atual,      bins=bins)[0] / len(atual)
    
    freq_ref   = np.where(freq_ref   == 0, 0.0001, freq_ref)
    freq_atual = np.where(freq_atual == 0, 0.0001, freq_atual)
    
    psi = np.sum((freq_atual - freq_ref) * np.log(freq_atual / freq_ref))
    return psi

print("3. MONITORAMENTO DE DRIFT (PSI):")
print(f"{'Feature':<20} {'PSI (sem drift)':>18} {'PSI (com drift)':>18} Status")
print("-"*75)

for feat in features:
    psi_ok    = calcular_psi(df_treino[feat].values, df_producao_ok[feat].values)
    psi_drift = calcular_psi(df_treino[feat].values, df_producao_drift[feat].values)
    
    def status(psi):
        if psi < 0.1:   return 'OK'
        elif psi < 0.25: return 'INVESTIGAR'
        else:            return 'RETREINAR!'
    
    print(f"{feat:<20} {psi_ok:>18.4f} {psi_drift:>18.4f} {status(psi_drift)}")

print()
print("INTERPRETACAO DO PSI:")
print("  < 0.10: mudanca insignificante -- modelo ainda valido")
print("  0.10-0.25: mudanca moderada -- investigar e monitorar")
print("  > 0.25: mudanca severa -- retreinar o modelo!")

In [ ]:
# C.2 -- Exercicios Deploy e Producao
print("="*65)
print("EXERCICIOS -- DEPLOY E MONITORAMENTO")
print("="*65)
print("""
E1: Qual a diferenca entre real-time endpoint e batch scoring?
    Para qual cenario bancario voce usaria cada um?

E2: O que e concept drift? Como voce detectaria que o modelo de
    propensao de credito degradou em producao?

E3: PSI de 'score_pj' = 0.35 no monitoramento mensal.
    O que isso significa e o que voce faz?

E4: Por que e importante versionar modelos (MLflow Model Registry)?
    O que acontece sem versionamento?

E5: Qual a diferenca entre data drift e concept drift?
    Cite um exemplo de cada no contexto bancario.
""")

print("GABARITOS:")
print("""
E1: Real-time endpoint: previsao em milissegundos sob demanda.
    Uso bancario: deteccao de fraude (decisao em < 100ms por transacao).
    
    Batch scoring: roda offline, processa grandes volumes, resultado em arquivo.
    Uso bancario: score de propensao diario para campanha de marketing.
    (Gera lista de 'clientes mais propensos ao produto X hoje')
    
    Regra: se a decisao precisa de resultado imediato -> real-time.
           se pode esperar horas/dias -> batch (mais barato e escalavel).

E2: Concept drift: a RELACAO entre X e Y muda.
    Ex: comportamento de pagamento mudou apos crise economica.
    Clientes com score 700 que antes tinham P(inadim)=5% agora tem 12%.
    
    Detectar: monitorar AUC/KS do modelo em producao mensalmente.
    Se AUC cai de 0.82 para 0.71 -> concept drift.
    Tambem: monitorar calibracao (probabilidade prevista vs taxa real).

E3: PSI=0.35 > 0.25 -> mudanca SEVERA na distribuicao de score_pj.
    Acao: investigar a causa (mudanca na politica de credito? crise?),
    retreinar o modelo com dados mais recentes, validar antes de implantar.
    Enquanto isso: monitorar mais frequentemente e alertar o time de negocio.

E4: Sem versionamento:
    - Nao da pra saber qual modelo esta em producao
    - Nao da pra fazer rollback se o novo modelo e pior
    - Nao da pra reproduzir resultados de analises passadas
    - Nao da pra auditar decisoes (BACEN pode exigir rastreabilidade)
    
    Com MLflow Model Registry: Staging (testando) -> Production (ativo).
    Cada versao tem params, metricas e codigo associados.
    Rollback em 1 comando se necessario.

E5: Data drift: distribuicao de X muda, relacao X->Y permanece.
    Exemplo: apos crise, renda media dos clientes caiu.
    'volume_mensal' mudou de distribuicao, mas o efeito na inadimplencia
    permanece o mesmo para o mesmo nivel de volume.
    Detectado por: PSI, KS test em cada feature.
    
    Concept drift: relacao X->Y muda, X pode estar estavel.
    Exemplo: pandemia mudou comportamento de pagamento.
    Clientes com mesmo perfil agora inadimplem mais.
    Detectado por: queda nas metricas (AUC, KS) em dados recentes.
""")

---
# MODULO D -- Prova Estimada 2026
## 40 Questoes: Conceituais + Praticas + Especificas da Vaga

**Instrucoes para simular:**
1. Feche os gabaritos
2. Cronometro: 90 minutos para as 40 questoes
3. Questoes praticas: rode o codigo, anote o resultado
4. Conceituais: responda sem consultar nada

**Estrutura estimada para 2026:**
- 15 questoes conceituais (0.2 pontos cada = 3.0 pts)
- 10 questoes praticas com codigo (0.5 pontos cada = 5.0 pts)
- 10 questoes de sistemas de recomendacao / experimentacao (0.2 pts = 2.0 pts)
- 5 questoes discursivas / caso (situacionais)


In [ ]:
# PROVA ESTIMADA 2026 -- QUESTOES CONCEITUAIS
print("="*70)
print("PROVA ESTIMADA 2026 -- SECAO 1: QUESTOES CONCEITUAIS (0.2 pts cada)")
print("="*70)

questoes_conceituais = [
    # Modulo 1 -- Estatistica
    ("Q01 (0.2pt) Assinale a alternativa CORRETA sobre p-valor:",
     ["(a) P-valor e a probabilidade de H0 ser verdadeira",
      "(b) P-valor < 0.05 garante que o efeito e relevante para o negocio",
      "(c) P-valor e a probabilidade de observar resultado tao extremo assumindo H0 verdadeira",
      "(d) P-valor de 0.03 significa 97% de chance do efeito ser real"],
     "(c)", "Definicao exata de p-valor. (a), (b) e (d) sao erros classicos."),

    # Modulo 2 -- Aprendizado
    ("Q02 (0.2pt) Sobre o Bias-Variance Tradeoff:",
     ["(a) Aumentar a complexidade do modelo sempre reduz o erro de validacao",
      "(b) Erro total = Bias + Variancia (sem termo de erro irredutivel)",
      "(c) Underfitting e caracterizado por alto bias e baixa variancia",
      "(d) Overfitting e caracterizado por alto bias e alta variancia"],
     "(c)", "Underfitting: modelo simples, erra sistematicamente (alto bias), estavel (baixa variancia)."),

    # Modulo 3 -- Regressao
    ("Q03 (0.2pt) R^2 de um modelo de regressao:",
     ["(a) Sempre diminui quando adicionamos variaveis irrelevantes",
      "(b) Pode assumir valores negativos",
      "(c) Nunca diminui com a adicao de variaveis ao modelo",
      "(d) E sempre igual ao R^2 ajustado quando p=1"],
     "(c)", "R^2 NUNCA diminui. R^2 ajustado pode diminuir com vars irrelevantes."),

    # Modulo 4 -- Classificacao
    ("Q04 (0.2pt) Sobre metricas de classificacao com base desbalanceada (95% negativos):",
     ["(a) Acuracia e a melhor metrica pois fornece o percentual de acertos",
      "(b) MAE e indicado quando ha muitos outliers na classe positiva",
      "(c) ROC-AUC e preferivel pois e insensivel ao desbalanceamento",
      "(d) Um modelo que preve sempre a classe negativa tem acuracia de 95% e e util"],
     "(c)", "ROC-AUC mede discriminacao e e mais robusta ao desbalanceamento. MAE e de regressao."),

    # Modulo 5 -- Cross-Validation
    ("Q05 (0.2pt) O padrao de cross-validation da sabatina do Itau confirmado empiricamente e:",
     ["(a) TimeSeriesSplit(n_splits=5)",
      "(b) KFold(n_splits=5, shuffle=True, random_state=42)",
      "(c) StratifiedKFold(n_splits=5)",
      "(d) KFold(n_splits=5, shuffle=False)"],
     "(d)", "Confirmado com dados reais: Q10 e Q11 da prova 2019 batem com KFold sem shuffle."),

    # Modulo 6 -- Regularizacao
    ("Q06 (0.2pt) Sobre Ridge com lambda tendendo ao infinito:",
     ["(a) Os coeficientes tendem ao infinito",
      "(b) O modelo se torna equivalente a uma regressao linear sem regularizacao",
      "(c) Os coeficientes tendem a zero",
      "(d) Lasso e Ridge tem comportamento identico com lambda grande"],
     "(c)", "Lambda->inf em Ridge: coefs->0. NUNCA->infinito. (Erro classico da Q14 de 2019!)"),

    # Modulo 8 -- Arvores
    ("Q07 (0.2pt) Principais causas de overfitting em Random Forest:",
     ["(a) Grande numero de arvores e taxa de aprendizado elevada",
      "(b) Arvores muito profundas e min_samples_leaf baixo",
      "(c) Grande numero de arvores e regularizacao L1 ausente",
      "(d) Taxa de aprendizado elevada e regularizacao L2 ausente"],
     "(b)", "RF: overfitting via max_depth alto. NAO tem learning_rate (isso e XGBoost!)."),

    # Modulo 9 -- SVM
    ("Q08 (0.2pt) Sobre SVM com kernel RBF e alto valor de Gamma:",
     ["(a) O modelo tem margem larga e alta regularizacao",
      "(b) Cada ponto tem influencia sobre regiao muito grande",
      "(c) O modelo e mais simples e propenso a underfitting",
      "(d) Cada ponto tem influencia sobre regiao muito pequena, tendendo a overfitting"],
     "(d)", "Alto Gamma = raio de influencia pequeno = modelo complexo = overfitting."),

    # Modulo 10 -- Redes Neurais
    ("Q09 (0.2pt) Uma rede neural com funcoes de ativacao lineares em TODAS as camadas:",
     ["(a) Tem maior capacidade que uma rede com ativacoes nao-lineares",
      "(b) Equivale matematicamente a uma unica camada linear",
      "(c) Resolve o problema de vanishing gradient",
      "(d) E recomendada para problemas de classificacao complexos"],
     "(b)", "Composicao de funcoes lineares = transformacao linear. Profundidade nao ajuda."),

    # Modulo 11 -- Clustering
    ("Q10 (0.2pt) O metodo average linkage calcula a distancia entre clusters como:",
     ["(a) Distancia entre os centroides dos clusters",
      "(b) Distancia minima entre qualquer par de pontos dos dois clusters",
      "(c) Media das distancias entre todos os pares de pontos dos dois clusters",
      "(d) Distancia maxima entre qualquer par de pontos dos dois clusters"],
     "(c)", "Average = media par-a-par. NAO e centroide (seria centroid linkage). (Q31 de 2019!)"),

    # Sistemas de Recomendacao
    ("Q11 (0.2pt) O problema de cold start em sistemas de recomendacao ocorre quando:",
     ["(a) A matriz usuario-item e muito densa (poucos valores ausentes)",
      "(b) Um novo usuario ou item e adicionado sem historico de interacoes",
      "(c) O modelo de matrix factorization converge para minimo local",
      "(d) Dois usuarios tem perfis identicos de preferencia"],
     "(b)", "Cold start: sem historico, filtragem colaborativa nao funciona. Solucao: conteudo/demografico."),

    # Experimentacao
    ("Q12 (0.2pt) O problema do 'peeking' em testes A/B consiste em:",
     ["(a) Usar amostra muito grande, gerando custo desnecessario",
      "(b) Checar os resultados continuamente e parar quando p<0.05, inflando falsos positivos",
      "(c) Usar a mesma amostra para treino e validacao do modelo",
      "(d) Nao calcular o tamanho de amostra antes do experimento"],
     "(b)", "Peeking: multiplas checagens inflam severamente a taxa de erro tipo 1."),

    # Metricas Recomendacao
    ("Q13 (0.2pt) NDCG@K difere de Precision@K pois:",
     ["(a) NDCG@K considera apenas o primeiro item relevante na lista",
      "(b) NDCG@K penaliza items relevantes em posicoes mais baixas da lista",
      "(c) NDCG@K e calculado sem necessidade de ground truth",
      "(d) NDCG@K e equivalente a Recall@K quando K e grande"],
     "(b)", "NDCG desconta por posicao: item relevante em pos 1 vale mais que em pos 5."),

    # PSI/Drift
    ("Q14 (0.2pt) Um PSI de 0.35 no monitoramento de uma feature indica:",
     ["(a) Mudanca insignificante -- modelo ainda valido",
      "(b) Mudanca moderada -- investigar mas nao e urgente",
      "(c) Mudanca severa -- modelo deve ser retreinado",
      "(d) Erro no calculo -- PSI nao pode ultrapassar 0.25"],
     "(c)", "PSI > 0.25 = mudanca severa na distribuicao. Retreinar o modelo."),

    # DiD
    ("Q15 (0.2pt) Diferencas em Diferencas (DiD) requer como premissa central:",
     ["(a) Randomizacao dos grupos tratado e controle",
      "(b) Tamanhos iguais nos grupos tratado e controle",
      "(c) Tendencias paralelas entre tratado e controle na ausencia do tratamento",
      "(d) Normalidade dos residuos em ambos os grupos"],
     "(c)", "Tendencias paralelas: sem o tratamento, ambos os grupos teriam evoluido identicamente."),
]

for i, (enunciado, alternativas, gabarito, explicacao) in enumerate(questoes_conceituais):
    print(f"\n{enunciado}")
    for alt in alternativas:
        print(f"  {alt}")

print()
print("="*70)
print("GABARITOS -- SECAO 1")
print("="*70)
for i, (enunciado, alternativas, gabarito, explicacao) in enumerate(questoes_conceituais):
    q_num = enunciado[:3]
    print(f"{q_num}: {gabarito} -- {explicacao}")

In [ ]:
# PROVA ESTIMADA 2026 -- QUESTOES PRATICAS
print("="*70)
print("PROVA ESTIMADA 2026 -- SECAO 2: QUESTOES PRATICAS (0.5 pts cada)")
print("="*70)
print()
print("Use os arquivos CSV da prova na mesma pasta do notebook.")
print()

kf5 = __import__('sklearn.model_selection', fromlist=['KFold']).KFold(5, shuffle=False)
kf10= __import__('sklearn.model_selection', fromlist=['KFold']).KFold(10, shuffle=False)

from sklearn.linear_model import ElasticNet, Ridge, Lasso, LogisticRegression
from sklearn.svm import SVR, SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_validate, KFold
from sklearn.metrics import roc_auc_score, log_loss
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import numpy as np, pandas as pd, os

print("Q16 (0.5pt) Elastic Net regressao_Q1.csv -- alpha=1.0, l1_ratio=0.01, KFold5:")
if os.path.exists('regressao_Q1.csv'):
    df=pd.read_csv('regressao_Q1.csv'); X,y=df.drop('target',1),df['target']
    r=cross_validate(ElasticNet(alpha=1.0,l1_ratio=0.01),X,y,cv=kf5,
                      scoring='neg_mean_squared_error',return_train_score=True)
    print(f"  Treino MSE: {-r['train_score'].mean():.4f} | Val MSE: {-r['test_score'].mean():.4f}")
else: print("  [arquivo nao encontrado]")

print()
print("Q17 (0.5pt) SVR linear regressao_Q2.csv -- C=0.001, KFold5:")
if os.path.exists('regressao_Q2.csv'):
    df=pd.read_csv('regressao_Q2.csv'); X,y=df.drop('target',1),df['target']
    r=cross_validate(SVR(kernel='linear',C=0.001),X,y,cv=kf5,
                      scoring='neg_mean_squared_error',return_train_score=True)
    print(f"  Treino MSE: {-r['train_score'].mean():.0f} | Val MSE: {-r['test_score'].mean():.0f}")
else: print("  [arquivo nao encontrado]")

print()
print("Q18 (0.5pt) Logistica C=0.5, L2, classificacao_Q2.csv, KFold10:")
if os.path.exists('classificacao_Q2.csv'):
    df=pd.read_csv('classificacao_Q2.csv'); X,y=df.drop('target',1),df['target']
    r=cross_validate(LogisticRegression(C=0.5,max_iter=1000),X,y,cv=kf10,
                      scoring='roc_auc',return_train_score=True)
    print(f"  Treino AUC: {r['train_score'].mean():.4f} | Val AUC: {r['test_score'].mean():.4f}")
else: print("  [arquivo nao encontrado]")

print()
print("Q19 (0.5pt) RandomForest classificacao_Q1.csv -- n_est=100, depth=5, KFold5, AUC:")
if os.path.exists('classificacao_Q1.csv'):
    df=pd.read_csv('classificacao_Q1.csv'); X,y=df.drop('target',1),df['target']
    r=cross_validate(RandomForestClassifier(n_estimators=100,max_depth=5,random_state=42),
                      X,y,cv=kf5,scoring='roc_auc',return_train_score=True)
    print(f"  Treino AUC: {r['train_score'].mean():.4f} | Val AUC: {r['test_score'].mean():.4f}")
else: print("  [arquivo nao encontrado]")

print()
print("Q20 (0.5pt) Ridge alpha=5.0, regressao_Q1.csv, KFold5, MSE treino e val:")
if os.path.exists('regressao_Q1.csv'):
    df=pd.read_csv('regressao_Q1.csv'); X,y=df.drop('target',1),df['target']
    r=cross_validate(Ridge(alpha=5.0),X,y,cv=kf5,
                      scoring='neg_mean_squared_error',return_train_score=True)
    print(f"  Treino MSE: {-r['train_score'].mean():.4f} | Val MSE: {-r['test_score'].mean():.4f}")
else: print("  [arquivo nao encontrado]")

print()
print("Q21 (0.5pt) SVM rbf C=1.0 gamma=0.1 -- classificacao_Q2.csv, KFold5, AUC:")
if os.path.exists('classificacao_Q2.csv'):
    df=pd.read_csv('classificacao_Q2.csv'); X,y=df.drop('target',1),df['target']
    pipe=Pipeline([('sc',StandardScaler()),('svm',SVC(kernel='rbf',C=1.0,gamma=0.1,probability=True))])
    r=cross_validate(pipe,X,y,cv=kf5,scoring='roc_auc',return_train_score=True)
    print(f"  Treino AUC: {r['train_score'].mean():.4f} | Val AUC: {r['test_score'].mean():.4f}")
else: print("  [arquivo nao encontrado]")

print()
print("Q22 (0.5pt) Arvore entropy sem poda -- classificacao_Q1.csv, KFold10, Log Loss:")
if os.path.exists('classificacao_Q1.csv'):
    df=pd.read_csv('classificacao_Q1.csv'); X,y=df.drop('target',1),df['target']
    r=cross_validate(DecisionTreeClassifier(criterion='entropy'),X,y,cv=kf10,
                      scoring='neg_log_loss',return_train_score=True)
    print(f"  Treino LL: {-r['train_score'].mean():.4f} | Val LL: {-r['test_score'].mean():.4f}")
else: print("  [arquivo nao encontrado]")

print()
print("Q23 (0.5pt) Lasso alpha=0.5 -- regressao_Q1.csv, KFold5, MSE:")
if os.path.exists('regressao_Q1.csv'):
    df=pd.read_csv('regressao_Q1.csv'); X,y=df.drop('target',1),df['target']
    r=cross_validate(Lasso(alpha=0.5),X,y,cv=kf5,
                      scoring='neg_mean_squared_error',return_train_score=True)
    print(f"  Treino MSE: {-r['train_score'].mean():.4f} | Val MSE: {-r['test_score'].mean():.4f}")
else: print("  [arquivo nao encontrado]")

print()
print("Q24 (0.5pt) Clustering hierarquico agrupamento.csv -- complete linkage, 3 grupos:")
if os.path.exists('agrupamento.csv'):
    from scipy.cluster.hierarchy import linkage, fcluster
    df=pd.read_csv('agrupamento.csv')
    Z=linkage(df.values,method='complete')
    for d in sorted(set(Z[:,2])):
        ng=len(set(fcluster(Z,d,criterion='distance')))
        if ng==3:
            print(f"  Limiar para 3 grupos (complete): {d:.4f}")
            break
else: print("  [arquivo nao encontrado]")

print()
print("Q25 (0.5pt) Calcule Precision@3 e NDCG@3 dado:")
print("  Recomendados (em ordem): [prod_A, prod_B, prod_C]")
print("  Relevantes reais:        {prod_A, prod_C, prod_D}")
rec = ['prod_A','prod_B','prod_C']
rel = {'prod_A','prod_C','prod_D'}
k=3
hits = [1 if r in rel else 0 for r in rec]
prec_k = sum(hits)/k
dcg  = sum(h/math.log2(i+2) for i,h in enumerate(hits))
idcg = sum(1/math.log2(i+2) for i in range(min(len(rel),k)))
ndcg_k = dcg/idcg
print(f"  Precision@3 = {prec_k:.4f}")
print(f"  NDCG@3      = {ndcg_k:.4f}")
print(f"  Explicacao: prod_A (pos1) e prod_C (pos3) sao relevantes.")
print(f"  NDCG penaliza prod_C por estar na pos3 em vez da pos1.")

In [ ]:
# PROVA ESTIMADA 2026 -- QUESTOES SITUACIONAIS
print("="*70)
print("PROVA ESTIMADA 2026 -- SECAO 3: QUESTOES SITUACIONAIS (0.5 pts cada)")
print("="*70)
print("""
Q26 (0.5pt) CASO -- Modelo de propensao para produto de investimento:
  Voce treinou um XGBoost. AUC no treino = 0.94. AUC no teste = 0.76.
  Seu colega sugere que e overfitting. Voce discorda e sugere data leakage.
  
  (a) Como voce distinguiria overfitting de data leakage?
  (b) Qual feature voce investigaria primeiro e por que?
  (c) Apos confirmado leakage, como voce corrigiria?

Q27 (0.5pt) CASO -- Teste A/B de nova tela de onboarding:
  Seu gestor quer parar o teste apos 2 dias porque 'ja apareceu p=0.03'.
  N atual = 800 por grupo. N necessario calculado = 3.200 por grupo.
  
  (a) O que voce diria ao gestor?
  (b) Que criterio voce proporia para decisao?
  (c) Quais metricas de guardrail voce monitoraria?

Q28 (0.5pt) CASO -- Sistema de recomendacao para novo cliente PJ:
  Empresa com 3 dias de conta, CNAE = 4711-3/01 (supermercado).
  Sem historico de transacoes ou produtos.
  
  Qual sua abordagem para recomendar o primeiro produto?
  Detalhe: tipo de modelo, features usadas, metrica de avaliacao.

Q29 (0.5pt) CASO -- PSI de 0.31 na feature 'volume_mensal':
  Monitoramento mensal do modelo de churn detectou PSI=0.31.
  
  (a) O que isso indica?
  (b) Que acoes voce tomaria imediatamente?
  (c) Como voce validaria se o modelo ainda e util antes de retreinar?

Q30 (0.5pt) CASO -- Comunicacao executiva:
  Seu modelo de propensao para antecipacao de recebiveis tem:
  AUC=0.81, Lift@10%=2.8x, Precision@20%=0.34
  
  Explique em 3 frases para o diretor comercial o que esses numeros significam
  e qual o impacto esperado se aplicar o modelo na proxima campanha.
""")

print("GABARITOS -- SECAO 3:")
print("""
Q26:
  (a) Overfitting: treino bom, validacao ruim, mas piora GRADUALMENTE
      com dados mais novos. Leakage: performance perfeita no treino,
      colapso total em dados producao (nao vistos no treino).
      Testar: validacao com janela temporal futura -- leakage colapsa,
      overfitting apenas degrada.
  
  (b) Feature com importancia inesperadamente alta (feature importance).
      Ex: 'data_do_investimento' ou 'saldo_apos_aplicacao' -- seriam leakage obvio.
      Qualquer feature que so existe DEPOIS que o cliente investiu.
  
  (c) Remover features com leakage. Recriar o dataset com split temporal correto:
      treino apenas com dados ANTERIORES ao evento. Re-treinar. Re-avaliar.

Q27:
  (a) 'O teste nao atingiu o tamanho de amostra calculado.
      Parar agora com p=0.03 e peeking -- a taxa de falso positivo real
      pode ser 20-30%, nao 5%. Precisamos de 3.200 por grupo, temos 800.'
  
  (b) Continuar ate atingir 3.200 por grupo.
      Criterio de decisao: p < 0.05 E IC 95% completamente acima de 0
      E efeito pratico relevante (>= 1pp de conversao).
  
  (c) Metricas de guardrail: taxa de abandono do onboarding (nao pode subir),
      tempo de sessao (nao pode cair), NPS no feedback imediato (nao pode cair),
      taxa de erros tecnicos (nao pode subir).

Q28:
  Abordagem de cold start com modelo de propensao + regras de negocio:
  Features: CNAE (supermercado), porte, regiao, data de constituicao.
  Modelo: logistica ou XGBoost treinado em clientes SIMILARES (mesmo segmento)
  que ja tem historico -- prevê P(contrata produto X) com features de cadastro.
  Adicionar regra: CNAE 4711 -> priorizar maquininha, folha de pagamento, Pix.
  Metrica: Precision@3 (dos 3 produtos recomendados, quantos sao aceitos).
  Baseline: produtos mais populares do segmento (supermercado).

Q29:
  (a) PSI=0.31 > 0.25: mudanca SEVERA na distribuicao do volume mensal.
      Os dados de producao sao muito diferentes do que o modelo viu no treino.
      
  (b) Acoes imediatas: alertar o time de negocio, investigar a causa
      (crise economica? sazonalidade? mudanca na base de clientes?),
      aumentar frequencia de monitoramento, avaliar impacto nas decisoes.
  
  (c) Antes de retreinar: medir AUC/KS em dados recentes (ultimos 30-60 dias).
      Se AUC caiu de 0.80 para < 0.70 -> retreinar urgente.
      Se AUC ainda e aceitavel (> 0.75) -> retreinar no proximo ciclo programado.
      Monitorar calibracao: taxa real de churn vs probabilidade prevista.

Q30:
  'O modelo identifica com boa precisao os clientes mais propensos a contratar
  antecipacao de recebiveis: ao abordar apenas os 10% mais propensos, a taxa
  de aceite e 2.8 vezes maior que uma campanha aleatoria, reduzindo custo por
  conversao em 64%. Em termos praticos: com 50.000 clientes elegíveis e
  custo de R$20 por abordagem, o modelo economiza cerca de R$640k por campanha
  mantendo o mesmo volume de contratos.'
""")

In [ ]:
# GABARITO FINAL E PONTUACAO
print("="*70)
print("GABARITO FINAL -- PROVA ESTIMADA 2026")
print("="*70)
print()

gabaritos_conceituais = {
    'Q01':'(c)', 'Q02':'(c)', 'Q03':'(c)', 'Q04':'(c)', 'Q05':'(d)',
    'Q06':'(c)', 'Q07':'(b)', 'Q08':'(d)', 'Q09':'(b)', 'Q10':'(c)',
    'Q11':'(b)', 'Q12':'(b)', 'Q13':'(b)', 'Q14':'(c)', 'Q15':'(c)',
}

print("SECAO 1 -- CONCEITUAIS (0.2 pts cada, total: 3.0 pts)")
for q, g in gabaritos_conceituais.items():
    print(f"  {q}: {g}")

print()
print("SECAO 2 -- PRATICAS (0.5 pts cada, total: 5.0 pts)")
print("  Q16-Q25: rode o codigo e compare com os valores obtidos")
print("  Padrao: KFold(n_splits=K, shuffle=False). neg_* x (-1).")

print()
print("SECAO 3 -- SITUACIONAIS (0.5 pts cada, total: 2.5 pts)")
print("  Q26-Q30: avalie por criterio de negocio + tecnico")

print()
print("="*70)
print("PONTUACAO TOTAL: 10.5 pontos possiveis")
print("  >= 8.0: APROVADO com margem")
print("  >= 6.5: APROVADO (depende do corte da turma)")
print("  < 6.5:  revisar os modulos com mais erros")
print()
print("TEMAS COM MAIOR PESO NESTA PROVA ESTIMADA:")
print("  Cross-validation (Q05, Q16-Q25): fundamental")
print("  Regularizacao e Ridge/Lasso (Q06, Q20, Q23): importante")
print("  Classificacao e metricas (Q04, Q18, Q19, Q22): importante")
print("  Sistemas de recomendacao (Q11, Q13, Q25, Q28): especifico da vaga")
print("  Experimentacao (Q12, Q15, Q27): especifico da vaga")
print("  Drift e producao (Q14, Q29): crescente em 2026")
print()
print("DIFERENCIAIS QUE SEPARAM CANDIDATOS EM 2026 vs 2019:")
print("  1. Saber calcular e interpretar metricas de recomendacao (NDCG, Precision@K)")
print("  2. Saber estruturar um teste A/B do zero (tamanho, guardrail, peeking)")
print("  3. Entender monitoramento de drift em producao (PSI, concept drift)")
print("  4. Comunicar resultado de modelo em linguagem de negocio (Q30)")
print("  5. Identificar data leakage (Q26) -- nao era cobrado em 2019")